# Inspect One Compact Event Batch - v7

Load one v7 compact event sample-cache batch, inspect tensor shapes and byte values, optionally restore a checkpoint, render a Keras-style model summary/torchview graph, run one masked-event inference, and calculate one reconstruction loss.

v7 masks whole event tokens for the encoder. The decoder reconstructs masked event bytes only; header bytes are visible to the encoder and are not reconstructed.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v7"
WORKSPACE = None  # Leave None to auto-detect local repo or workstation runtime root.
SAMPLE_CACHE_ROOT = Path(r"D:\market-data\prepared\event_sample_cache")
BATCH_SIZE = 8
EVENTS_PER_CHUNK = 128
DEVICE = "cuda"
USE_AMP = False

# Paste a checkpoint path, or leave empty to auto-load the newest checkpoint under the v7 runtime tree.
CHECKPOINT_PATH = ""
RUNTIME_ROOT = Path(r"D:\TradingML\runtimes\masked_event_model\v7\pretrain")
ARCHITECTURE_OUTPUT_DIR = Path(r"D:\TradingML\runtimes\masked_event_model\v7\notebook_artifacts\inspect_architecture")


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch
from IPython.display import Image, Markdown, SVG, display


def resolve_workspace(explicit):
    if explicit:
        path = Path(explicit)
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.extend([
        Path(r"D:\TradingCodes\quant-research-workbench"),
        Path(r"D:\TradingML\codes\masked_event_model\v7"),
        Path(r"\\DESKTOP-SAAI85T\Workstation-D\TradingML\codes\masked_event_model\v7"),
    ])
    for path in candidates:
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    raise FileNotFoundError("Could not find a workspace root containing research/masked_event_model/v7.")


WORKSPACE = resolve_workspace(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.masked_event_model.v7.config import LossConfig, MaskConfig, ModelConfig
from research.masked_event_model.v7.losses import masked_event_bce_loss, pack_bits
from research.masked_event_model.v7.masking import build_event_masks
from research.masked_event_model.v7.model import EVENT_BYTES, HEADER_BYTES, EventTokenMaskedAutoencoder
from research.mlops.event_sample_cache import EventSampleCacheDataConfig, discover_event_sample_shards, iter_event_sample_cache_epoch_batches, resolve_event_sample_cache_root

np.set_printoptions(precision=5, suppress=True)
torch.set_printoptions(precision=5, sci_mode=False)
torch.set_float32_matmul_precision("high")

device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
cache_root = resolve_event_sample_cache_root(SAMPLE_CACHE_ROOT)
print("workspace:", WORKSPACE)
print("cache_root:", cache_root)
print("device:", device)


In [ ]:
cache_config = EventSampleCacheDataConfig(
    cache_root=SAMPLE_CACHE_ROOT,
    split="train",
    batch_size=BATCH_SIZE,
    events_per_chunk=EVENTS_PER_CHUNK,
    seed=17,
    prefetch_shards=1,
    max_shards=1,
    shuffle_records=False,
    drop_last=False,
)
shards = discover_event_sample_shards(cache_config)
batch = next(iter_event_sample_cache_epoch_batches(cache_config, epoch=1, shards=shards))
print("shard count:", len(shards))
print("first shard:", shards[0].path)
print("batch keys:", sorted(batch.keys()))
for key, value in batch.items():
    if torch.is_tensor(value):
        min_value = int(value.min()) if value.numel() else "n/a"
        max_value = int(value.max()) if value.numel() else "n/a"
        print(f"{key:24s} shape={tuple(value.shape)} dtype={value.dtype} min={min_value} max={max_value}")
    else:
        print(f"{key:24s} type={type(value).__name__} value={value}")


In [ ]:
row = 0
header = batch["header_uint8"][row].numpy()
events = batch["events_uint8"][row].numpy()
print("shard_index:", batch.get("shard_index"))
print("origin_timestamp_ns:", int(batch["origin_timestamp_ns"][row]))
display(pl.DataFrame({"header_byte_index": list(range(len(header))), "uint8": header.tolist()}))
display(pl.DataFrame(events[:20], schema=[f"byte_{idx:02d}" for idx in range(events.shape[1])], orient="row"))


In [ ]:
def latest_checkpoint(runtime_root):
    candidates = list(runtime_root.glob("*/checkpoints/latest.pt")) + list(runtime_root.glob("*/checkpoints/checkpoint_latest.pt")) + list(runtime_root.glob("*/checkpoints/*.pt"))
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)

checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(RUNTIME_ROOT)
checkpoint = None
checkpoint_config = None
if checkpoint_path and checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    checkpoint_config = checkpoint.get("config")
    print("loaded checkpoint:", checkpoint_path)
    print("checkpoint step:", checkpoint.get("step"))
else:
    print("No checkpoint loaded; using fresh randomly initialized model.")

model_config = ModelConfig(**{k: v for k, v in (checkpoint_config or {}).get("model", {}).items() if k in ModelConfig.__dataclass_fields__}) if checkpoint_config else ModelConfig()
mask_config = MaskConfig(**{k: v for k, v in (checkpoint_config or {}).get("masks", {}).items() if k in MaskConfig.__dataclass_fields__}) if checkpoint_config else MaskConfig()
loss_config = LossConfig(**{k: v for k, v in (checkpoint_config or {}).get("losses", {}).items() if k in LossConfig.__dataclass_fields__}) if checkpoint_config else LossConfig()
model = EventTokenMaskedAutoencoder(events_per_chunk=EVENTS_PER_CHUNK, config=model_config).to(device)
if checkpoint is not None:
    model.load_state_dict(checkpoint.get("model", checkpoint.get("model_state_dict", checkpoint)), strict=True)
model.eval()
print(f"model parameters={sum(p.numel() for p in model.parameters()):,}")
print("model_config:", model_config)


In [ ]:
# Keras-style model summary and torchview graph.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.

ARCHITECTURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_header = torch.zeros((1, HEADER_BYTES), dtype=torch.uint8, device=device)
summary_events = torch.zeros((1, EVENTS_PER_CHUNK, EVENT_BYTES), dtype=torch.uint8, device=device)

class TorchInfoSummaryWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, header_uint8, events_uint8):
        masks = build_event_masks(events_uint8, mask_config)
        output = self.model(header_uint8, events_uint8, masks, mask_config=None)
        return torch.cat([output.event_bit_probs.flatten(), output.chunk_embedding.flatten()]).unsqueeze(0)

summary_model = TorchInfoSummaryWrapper(model).to(device)
try:
    from torchinfo import summary
    model_summary = summary(
        summary_model,
        input_data=(summary_header, summary_events),
        depth=8,
        col_names=("input_size", "output_size", "num_params", "trainable"),
        verbose=0,
    )
    summary_text = str(model_summary)
except Exception as exc:
    summary_text = f"torchinfo summary failed: {exc}\n\n{model}"

summary_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_summary.txt"
summary_path.write_text(summary_text, encoding="utf-8")
display(Markdown("```text\n" + summary_text[:50000] + ("\n... truncated ..." if len(summary_text) > 50000 else "") + "\n```"))
print("summary_path:", summary_path)

try:
    from torchview import draw_graph
    graph = draw_graph(summary_model, input_data=(summary_header, summary_events), expand_nested=True, depth=5, save_graph=False)
    graph.visual_graph.attr(dpi="180")
    png_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_torchview.png"
    svg_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_torchview.svg"
    graph.visual_graph.render(filename=str(png_path.with_suffix("")), directory=str(png_path.parent), format="png", cleanup=True)
    graph.visual_graph.render(filename=str(svg_path.with_suffix("")), directory=str(svg_path.parent), format="svg", cleanup=True)
    print("torchview_png:", png_path)
    print("torchview_svg:", svg_path)
    if png_path.exists():
        display(Image(filename=str(png_path)))
except Exception as exc:
    print("torchview graph failed:", repr(exc))


In [ ]:
def move_tensor_batch(batch_dict, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch_dict.items()}

model.eval()
device_batch = move_tensor_batch(batch, device)
masks = build_event_masks(device_batch["events_uint8"], mask_config)
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
    output = model(device_batch["header_uint8"], device_batch["events_uint8"], masks, mask_config=None)
    result = masked_event_bce_loss(output, loss_config, include_diagnostics=True)
    chunk_embedding = model.encode(device_batch["header_uint8"], device_batch["events_uint8"])
    event_embeddings = model.encode_events(device_batch["header_uint8"], device_batch["events_uint8"])

print("loss:", float(result.loss.detach().cpu()))
print("metrics:", json.dumps(result.metrics, indent=2, sort_keys=True))
print("event probs:", tuple(output.event_bit_probs.shape))
print("masked_event_indices:", tuple(output.masked_event_indices.shape))
print("target_events_uint8:", tuple(output.target_events_uint8.shape))
print("chunk_embedding:", tuple(chunk_embedding.shape))
print("event_embeddings:", tuple(event_embeddings.shape))


In [ ]:
sample = 0
event_indices = output.masked_event_indices.detach().cpu()
event_probs = output.event_bit_probs.detach().cpu()
event_pred = pack_bits(event_probs >= 0.5).cpu() if event_probs.numel() else torch.empty(0, dtype=torch.long)

rows = []
for event_idx, pred_bytes in zip(event_indices[sample], event_pred[sample]):
    event_idx = int(event_idx)
    for byte_idx, pred in enumerate(pred_bytes):
        target = int(batch["events_uint8"][sample, event_idx, byte_idx])
        rows.append({"sample": sample, "event": event_idx, "byte": byte_idx, "target_uint8": target, "pred_uint8": int(pred), "abs_error": abs(int(pred) - target)})
pl.DataFrame(rows).sort(["event", "byte"]).head(300)
